# Chapter 10 — Exercises: Worked Solutions

Worked solutions for Chapter 10 (Gas Processing and Glycol Systems).

**Exercises:**
1. TEG–water VLE and activity coefficients
2. Water dew point depression with TEG
3. MEG injection for hydrate inhibition

In [1]:
import importlib, subprocess, sys
try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False); ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
except Exception:
    try: import neqsim
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
print(f"NeqSim mode: {NEQSIM_MODE}")

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim


NeqSim mode: pip


In [2]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PALETTE = ["#2171b5", "#e6550d", "#31a354", "#756bb1", "#e7298a", "#66a61e"]
BLUE, ORANGE, GREEN, PURPLE, PINK, LIME = PALETTE
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.labelsize": 10, "axes.titlesize": 10, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "xtick.direction": "in", "ytick.direction": "in", "axes.linewidth": 0.6, "lines.linewidth": 1.2, "grid.linewidth": 0.3, "grid.alpha": 0.4, "savefig.dpi": 300, "figure.dpi": 150})
FIGURES_DIR = Path("../figures"); FIGURES_DIR.mkdir(exist_ok=True)
def save(fig, name): fig.savefig(str(FIGURES_DIR / name), dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig); print(f"Saved: {name}")
R = 8.314

SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

---
## Exercise 10.1 — TEG–Water VLE

**Problem:** Calculate the bubble point temperature for a TEG–water
mixture at 1 atm for TEG mole fractions from 0.05 to 0.95.

### Solution

In [3]:
x_teg = np.arange(0.05, 0.96, 0.05)
T_bp = []

for xt in x_teg:
    try:
        f = SystemSrkCPAstatoil(273.15 + 100.0, 1.01325)  # start at 100 C, 1 atm
        f.addComponent("TEG", float(xt))
        f.addComponent("water", float(1.0 - xt))
        f.setMixingRule(10)
        ops = ThermodynamicOperations(f)
        ops.bubblePointTemperatureFlash()
        T_bp.append(float(f.getTemperature()) - 273.15)
    except Exception:
        T_bp.append(np.nan)

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.plot(x_teg[:len(T_bp)], T_bp, 'o-', color=BLUE, markersize=3, linewidth=1.2)
ax.axhline(y=100, color="grey", linestyle=":", alpha=0.5)
ax.set_xlabel("$x_{TEG}$ (mol frac)"); ax.set_ylabel("Bubble point T (\u00b0C)")
ax.set_title("Exercise 10.1: TEG\u2013water Txy")
ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch10_ex01_teg_water_bp.png")

Saved: fig_ch10_ex01_teg_water_bp.png


**Answer:** The bubble point rises with increasing TEG concentration
because TEG (bp ~285\u00b0C) is much less volatile than water (bp 100\u00b0C).
The strong hydrogen bonding between TEG and water, captured by CPA,
causes negative deviations from Raoult's law, which CPA models through
the association cross-interaction.

---
## Exercise 10.2 — Water Dew Point Depression

**Problem:** A wet natural gas at 70 bar contains 500 ppm water.
Calculate the water dew point temperature. How much must TEG reduce
the water content to meet a -18\u00b0C dew point spec?

### Solution

In [4]:
# Water dew point for different water contents
water_ppm = [500, 300, 200, 100, 50, 20, 10]
T_dewpoint = []

for ppm in water_ppm:
    try:
        y_w = ppm / 1e6
        f = SystemSrkCPAstatoil(273.15 + 20.0, 70.0)
        f.addComponent("methane", 0.90 * (1 - y_w))
        f.addComponent("ethane", 0.05 * (1 - y_w))
        f.addComponent("propane", 0.03 * (1 - y_w))
        f.addComponent("CO2", 0.02 * (1 - y_w))
        f.addComponent("water", y_w)
        f.setMixingRule(10)
        f.setMultiPhaseCheck(True)
        ops = ThermodynamicOperations(f)
        ops.waterDewPointTemperatureFlash()
        T_dewpoint.append(float(f.getTemperature()) - 273.15)
    except Exception:
        T_dewpoint.append(np.nan)

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.plot(water_ppm[:len(T_dewpoint)], T_dewpoint, 'o-', color=BLUE, markersize=4, linewidth=1.2)
ax.axhline(y=-18, color="red", linestyle="--", alpha=0.5, linewidth=0.8)
ax.annotate("Spec: \u221218\u00b0C", xy=(200, -15), fontsize=7, color="red")
ax.set_xlabel("Water content (ppm)"); ax.set_ylabel("Water dew point (\u00b0C)")
ax.set_title("Exercise 10.2: Water dew point vs content")
ax.grid(True, alpha=0.3); ax.invert_xaxis()
fig.tight_layout()
save(fig, "fig_ch10_ex02_water_dewpoint.png")

Saved: fig_ch10_ex02_water_dewpoint.png


**Answer:** To achieve a -18\u00b0C water dew point, the gas water content
must be reduced to approximately 20–50 ppm, depending on composition
and pressure. This requires a lean TEG concentration of ~99 wt% or
higher in the contactor.

---
## Exercise 10.3 — MEG Activity Coefficient

**Problem:** Calculate the activity coefficient of water in a
MEG–water mixture as a function of MEG concentration at 25\u00b0C.

### Solution

In [5]:
x_meg = np.arange(0.05, 0.96, 0.05)
gamma_water = []

for xm in x_meg:
    try:
        f = SystemSrkCPAstatoil(273.15 + 25.0, 1.01325)
        f.addComponent("MEG", float(xm))
        f.addComponent("water", float(1.0 - xm))
        f.setMixingRule(10)
        ops = ThermodynamicOperations(f)
        ops.TPflash()
        f.initProperties()
        # Get fugacity coefficient and compute activity coeff
        phi_w = float(f.getPhase("aqueous").getComponent("water").getFugacityCoefficient())
        # For reference: pure water at same T, P
        fp = SystemSrkCPAstatoil(273.15 + 25.0, 1.01325)
        fp.addComponent("water", 1.0)
        fp.setMixingRule(10)
        opsp = ThermodynamicOperations(fp)
        opsp.TPflash()
        fp.initProperties()
        phi_pure = float(fp.getPhase(0).getComponent("water").getFugacityCoefficient())
        gamma = phi_w / phi_pure
        gamma_water.append(gamma)
    except Exception:
        gamma_water.append(np.nan)

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.plot(x_meg[:len(gamma_water)], gamma_water, 'o-', color=BLUE, markersize=3, linewidth=1.2, label="CPA")
ax.axhline(y=1.0, color="grey", linestyle=":", alpha=0.5)
ax.set_xlabel("$x_{MEG}$ (mol frac)"); ax.set_ylabel(r"$\gamma_{water}$")
ax.set_title("Exercise 10.3: Water activity in MEG")
ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch10_ex03_meg_activity.png")

Saved: fig_ch10_ex03_meg_activity.png


**Answer:** The activity coefficient of water in MEG–water is close to
1 at low MEG concentrations (near-ideal) and decreases slightly at
higher MEG because of the strong hydrogen bonding between MEG and water.
CPA captures these interactions through the cross-association between
MEG (2B) and water (4C) sites.